# Pandas from First Principles
## Notebook 4: String & DateTime Operations

**Series overview:**
| Notebook | Topic |
|----------|-------|
| 1 | Series & DataFrames |
| 2 | Indexing & Selection |
| 3 | Reading, Cleaning & Exporting Data |
| **4** | **String & DateTime Operations** ← you are here |
| 5 | Sorting & Ranking |

---

## 1. Why String & DateTime Operations Matter

Real-world datasets are messy. Column values that *look* like data often contain:

- **Extra whitespace** – `'  Alice '` instead of `'Alice'`
- **Inconsistent casing** – `'ENGINEERING'`, `'Engineering'`, `'engineering'` all mean the same thing
- **Dates stored as plain text** – `'2024-01-15'` is a string, not a date

Before you can analyse anything, you must **clean** this raw text data.  
Pandas gives you two powerful accessors for this:

| Accessor | Purpose | Applies to |
|----------|---------|------------|
| `.str` | String / text operations | `object` dtype columns |
| `.dt` | Date / time operations | `datetime64` dtype columns |

This notebook covers both from scratch.

In [ ]:
import pandas as pd

print("Pandas version:", pd.__version__)

---
## 2. String Operations — the `.str` Accessor

### What is it?
The `.str` accessor lets you apply Python string methods **element-wise** to every value in a Series — without writing a loop.

### Why do we need it?
Normal Python string methods (e.g. `'hello'.upper()`) work on a single string.  
When you have a whole column of strings in a DataFrame, you need `.str` to apply the same method to **every row at once**.

### Dataset we will use throughout Section 2

In [ ]:
employees = pd.DataFrame({
    'name':       ['  alice sharma ', 'BOB MEHTA', 'carol singh  ', '  DAVE kumar'],
    'email':      ['alice@gmail.com', 'bob@yahoo.com', 'carol@gmail.com', 'dave@outlook.com'],
    'department': ['engineering', 'MARKETING', 'Engineering', 'hr'],
    'phone':      ['9876543210', '8765432109', 'N/A', '7654321098'],},
    index=        [1,2,3,4])

display(employees)
print("\nColumn dtypes:")
display(employees.dtypes.to_frame(name='DataType'))

---
### 2.1 Removing Whitespace — `.str.strip()`, `.str.lstrip()`, `.str.rstrip()`

**What:** Remove leading and/or trailing whitespace characters (spaces, tabs, newlines).

| Method | Removes whitespace from |
|--------|------------------------|
| `.str.strip()` | **both** ends |
| `.str.lstrip()` | **left** end only |
| `.str.rstrip()` | **right** end only |

**Syntax:** `series.str.strip()`  
**Returns:** A new Series with cleaned strings (original is not modified).

**Best practice:** Almost always use `.str.strip()` first — whitespace is invisible and causes silent bugs (e.g. two rows that look the same but don't match in a filter).

In [ ]:
# Notice the 'name' column has leading/trailing spaces
print("Before strip:")
print(employees['name'].tolist())

# .str.strip() removes whitespace from both sides
cleaned_names = employees['name'].str.strip()

print("\nAfter .str.strip():")
print(cleaned_names.tolist())

# Demonstrate lstrip vs rstrip
print("\nAfter .str.lstrip() (left only):")
print(employees['name'].str.lstrip().tolist())

print("\nAfter .str.rstrip() (right only):")
print(employees['name'].str.rstrip().tolist())

---
### 2.2 Case Conversion — `.str.lower()`, `.str.upper()`, `.str.title()`

**What:** Standardise the casing of text data.

| Method | Effect | Example |
|--------|--------|---------|
| `.str.lower()` | all lowercase | `'BOB'` → `'bob'` |
| `.str.upper()` | ALL UPPERCASE | `'bob'` → `'BOB'` |
| `.str.title()` | Title Case | `'alice sharma'` → `'Alice Sharma'` |

**Returns:** A new Series with converted strings.

**Best practice:** Use `.str.lower()` before any comparison or grouping so that `'ENGINEERING'` and `'engineering'` are treated as the same value.

In [ ]:
# department column has mixed casing — let's standardise it
print("Original department column:")
print(employees['department'].tolist())

print("\n.str.lower() →", employees['department'].str.lower().tolist())
print(".str.upper() →", employees['department'].str.upper().tolist())

# Title case is ideal for display names
stripped_names = employees['name'].str.strip()
print("\nName after strip + title case:")
print(stripped_names.str.title().tolist())

---
### 2.3 Replacing Substrings — `.str.replace(old, new)`

**What:** Replace every occurrence of a substring (or regex pattern) with a new string.

**Syntax:** `series.str.replace(old, new, regex=False)`

**Parameters:**
- `old` – substring to find
- `new` – what to put in its place
- `regex=False` – treat `old` as a plain string, not a regex pattern (recommended for beginners)

**Returns:** A new Series with replacements applied.

**Common mistake:** Forgetting `regex=False` can cause unexpected results if your search string contains regex special characters like `.` or `*`.

In [ ]:
# Replace 'N/A' text in the phone column with an empty string
print("Original phone column:")
print(employees['phone'].tolist())

cleaned_phone = employees['phone'].str.replace('N/A', 'Unknown', regex=False)
print("\nAfter replacing 'N/A' with 'Unknown':")
print(cleaned_phone.tolist())

# Another example: standardise department names
dept_fixed = employees['department'].str.lower().str.replace('hr', 'human resources', regex=False)
print("\nDepartment after lower + replace 'hr':")
print(dept_fixed.tolist())

---
### 2.4 Checking for Substrings — `.str.contains(pattern)`

**What:** Returns a boolean Series — `True` if the string contains the pattern, `False` otherwise.

**Syntax:** `series.str.contains(pattern, case=True, na=False)`

**Parameters:**
- `pattern` – substring or regex to look for
- `case=True` – case-sensitive by default; set to `False` to ignore case
- `na=False` – what value to return for NaN rows (default is `NaN`; using `False` keeps things clean)

**Returns:** A boolean Series — perfect for filtering rows.

**Best practice:** Always set `na=False` (or `na=True`) explicitly so NaN rows don't cause errors downstream.

In [ ]:
# Which employees use Gmail?
is_gmail = employees['email'].str.contains('gmail', case=False, na=False)
print("Gmail mask (True/False per row):")
print(is_gmail.tolist())

# Use the boolean mask to filter the DataFrame
gmail_employees = employees[is_gmail]
print("\nEmployees with Gmail:")
print(gmail_employees[['name', 'email']])

---
### 2.5 Prefix & Suffix Checks — `.str.startswith()` and `.str.endswith()`

**What:** Check whether each string starts or ends with a given substring.

**Syntax:**
```python
series.str.startswith(prefix, na=False)
series.str.endswith(suffix, na=False)
```

**Returns:** Boolean Series (same as `.str.contains`).

**When to use:** Great for checking file extensions, country codes, prefixes in IDs, etc.

In [ ]:
# Which emails end with '.com'?
ends_com = employees['email'].str.endswith('.com', na=False)
print("Ends with '.com':")
print(ends_com.tolist())

# Which phone numbers start with '98'?
starts_98 = employees['phone'].str.startswith('98', na=False)
print("\nPhone starts with '98':")
print(employees[['name', 'phone']][starts_98])

---
### 2.6 Splitting Strings — `.str.split(sep)` and Accessing Parts

**What:** Split each string into a list of parts using a separator character.

**Syntax:** `series.str.split(sep)`

**Parameters:**
- `sep` – the character to split on (e.g. `'@'`, `'-'`, `' '`)

**Returns:** A Series of **lists**.

**Accessing a specific part:**  
Chain another `.str[index]` to get the element at that position from every list.  
- `.str.split('@').str[0]` → part **before** `@`  
- `.str.split('@').str[1]` → part **after** `@`

In [ ]:
# Split the email on '@' to separate username from domain
split_result = employees['email'].str.split('@')
print("After split('@') — each value is now a list:")
print(split_result.tolist())

# Extract username (index 0) and domain (index 1)
employees['username'] = employees['email'].str.split('@').str[0]
employees['domain']   = employees['email'].str.split('@').str[1]

print("\nEmail parts extracted:")
display(employees[['email', 'username', 'domain']])

---
### 2.7 String Length — `.str.len()`

**What:** Returns the number of characters in each string.

**Syntax:** `series.str.len()`

**Returns:** A numeric Series of integer lengths.

**When to use:** Validate phone numbers, password length, detect unusually short/long entries.

In [ ]:
# Check the length of every phone number
employees['phone_length'] = employees['phone'].str.len()

print("Phone numbers and their lengths:")
display(employees[['name', 'phone', 'phone_length']])

# Flag invalid entries (a valid phone should have exactly 10 digits)
valid_phone_mask = employees['phone_length'] == 10
print("\nRows with valid 10-digit phone numbers:")
display(employees[valid_phone_mask][['name', 'phone']])

---
### 2.8 Chaining `.str` Methods

**What:** You can chain multiple `.str` calls in sequence — the output of each becomes the input of the next.

**Why:** Avoids creating many intermediate variables. Reads naturally from left to right.

**Pattern:**
```python
series.str.strip().str.lower().str.replace('old', 'new')
```

**Best practice:** Chain in a logical order — strip first, then case, then replace.

In [ ]:
# Full cleaning pipeline on the 'name' column in a single chain
employees['name_clean'] = (
    employees['name']
    .str.strip()          # remove leading/trailing whitespace
    .str.title()          # convert to Title Case
)

# Full cleaning pipeline on the 'department' column
employees['dept_clean'] = (
    employees['department']
    .str.strip()
    .str.lower()
)

print("Cleaned columns:")
print(employees[['name', 'name_clean', 'department', 'dept_clean']])

---
### 2.9 Handling NaN in String Operations

**What happens by default:** When a value in a column is `NaN`, most `.str` methods will return `NaN` for that row — they do **not** raise an error and do **not** skip the row silently.

This is actually useful: `NaN` stays `NaN`, so you can still identify missing values after cleaning.

**Important:** For `.str.contains()`, the default `na=NaN` can cause issues in boolean filters. Always pass `na=False` (treat NaN rows as non-matching) or `na=True`.

In [ ]:
import numpy as np

# Series with a NaN value
names_with_nan = pd.Series(['  alice ', 'BOB', None, '  carol  '])
print("Original series (with None):")
print(names_with_nan.tolist())

# .str.strip() — NaN stays NaN
print("\nAfter .str.strip():")
print(names_with_nan.str.strip().tolist())

# .str.contains() — na=False keeps boolean output safe
print("\n.str.contains('ali', na=False):")
print(names_with_nan.str.contains('ali', na=False).tolist())

print("\n.str.contains('ali', na=True):")
print(names_with_nan.str.contains('ali', na=True).tolist())

---
### ⚠️ Common Mistake: Calling Python String Methods Directly on a Series

**The mistake:** Forgetting the `.str` accessor and calling the method directly on the Series object.

```python
# WRONG — this calls the Series method, not the string method
employees['name'].upper()   # AttributeError!
employees['name'].strip()   # AttributeError!

# CORRECT — always use .str accessor
employees['name'].str.upper()
employees['name'].str.strip()
```

**Rule to remember:** Anything you'd normally call on a single Python string must go through `.str` when applied to a Pandas Series.

In [ ]:
# Demonstrating the correct usage
sample = pd.Series(['hello world', 'pandas rocks', 'data cleaning'])

# CORRECT
print("Correct — using .str accessor:")
print(sample.str.upper().tolist())

# WRONG — uncomment to see the error
# print(sample.upper())  # AttributeError: 'Series' object has no attribute 'upper'

---
## 3. DateTime Operations — the `.dt` Accessor

### What is it?
The `.dt` accessor lets you extract date and time components (year, month, day, hour, weekday …) from a **datetime-typed** Series element-wise.

### Why do we need it?
When dates are stored as plain strings (`'2024-01-15'`), Pandas treats them as text — you can't subtract two strings to get a duration, you can't filter by month, etc.  
After converting to `datetime64` dtype with `pd.to_datetime()`, the `.dt` accessor unlocks all date arithmetic.

### Dataset we will use throughout Section 3

In [ ]:
orders = pd.DataFrame({
    'order_id':      [1, 2, 3, 4, 5, 6],
    'order_date':    ['2024-01-15', '2024-02-10', '2024-01-28',
                      '2024-03-05', '2024-02-22', '2024-03-18'],
    'delivery_date': ['2024-01-18', '2024-02-14', '2024-02-01',
                      '2024-03-08', '2024-02-26', '2024-03-22'],
    'amount':        [1500, 3200, 800, 5400, 2100, 900],
})

display(orders)
print("\nColumn dtypes before conversion:")
display(orders.dtypes.to_frame(name='DataType'))

---
### 3.1 Converting Strings to Datetime — `pd.to_datetime()`

**What:** Parse string columns and convert them to `datetime64` dtype so Pandas can perform date arithmetic and use the `.dt` accessor.

**Syntax:** `pd.to_datetime(series, format=None, errors='raise')`

**Parameters:**
- `format` – optional; specify the string format for faster parsing (e.g. `'%Y-%m-%d'`)
- `errors='coerce'` – converts unparseable strings to `NaT` (Not a Time) instead of raising an error

**Returns:** A Series with dtype `datetime64[ns]`.

In [ ]:
# Convert both date columns from strings to datetime
orders['order_date']    = pd.to_datetime(orders['order_date'])
orders['delivery_date'] = pd.to_datetime(orders['delivery_date'])

print("Column dtypes AFTER conversion:")
display(orders.dtypes.to_frame())

print("\nFirst few rows:")
display(orders.head())

---
### 3.2 Extracting Date Parts — `.dt.year`, `.dt.month`, `.dt.day`

**What:** Once a column has `datetime64` dtype, you can extract individual components.

| Accessor | Returns | Example |
|----------|---------|--------|
| `.dt.year` | Year as integer | `2024` |
| `.dt.month` | Month number (1–12) | `1` for January |
| `.dt.day` | Day of month (1–31) | `15` |
| `.dt.date` | Python `date` object (no time part) | `2024-01-15` |

**Returns:** A numeric Series (or object Series for `.dt.date`).

In [ ]:
# Extract year, month, day from order_date
orders['order_year']  = orders['order_date'].dt.year
orders['order_month'] = orders['order_date'].dt.month
orders['order_day']   = orders['order_date'].dt.day
orders['order_date_only'] = orders['order_date'].dt.date

orders.index = [1,2,3,4,5,6]
print("Date parts extracted:")
display(orders)


---
### 3.3 Extracting Time Parts — `.dt.hour`, `.dt.minute`

**What:** Extract the time components from a datetime column.

| Accessor | Returns |
|----------|---------|
| `.dt.hour` | Hour (0–23) |
| `.dt.minute` | Minute (0–59) |
| `.dt.second` | Second (0–59) |

**Note:** If the original string had no time component (e.g. `'2024-01-15'`), the time defaults to `00:00:00` — so `.dt.hour` will return `0` for all rows.

In [ ]:
# Create a sample series WITH time information
timestamps = pd.Series([
    '2024-01-15 09:30:00',
    '2024-02-10 14:15:45',
    '2024-03-05 22:00:00',
])
timestamps = pd.to_datetime(timestamps)

print("Timestamps:")
display(timestamps.to_frame(name='DateTime'))

print("\n.dt.hour  :", timestamps.dt.hour.tolist())
print(".dt.minute:", timestamps.dt.minute.tolist())
print(".dt.second:", timestamps.dt.second.tolist())

# For our orders dataset (no time): hours will all be 0
print("\nHour in orders['order_date'] (no time info → all 0):")
print(orders['order_date'].dt.hour.tolist())

---
### 3.4 Day of Week — `.dt.weekday` and `.dt.day_name()`

**What:** Find out which day of the week each date falls on.

| Accessor | Returns | Notes |
|----------|---------|-------|
| `.dt.weekday` | Integer 0–6 | 0 = Monday, 6 = Sunday |
| `.dt.day_name()` | String | `'Monday'`, `'Tuesday'`, … |

**Why useful:** Analysing patterns like "most orders placed on Fridays" or "fewest deliveries on weekends".

In [ ]:
# Which day of the week was each order placed?
orders['weekday_num']  = orders['order_date'].dt.weekday
orders['weekday_name'] = orders['order_date'].dt.day_name()

print("Orders with weekday information:")
display(orders[['order_id', 'order_date', 'weekday_num', 'weekday_name']])

print("\nReminder: 0=Monday, 1=Tuesday, ..., 6=Sunday")

---
### 3.5 Date Arithmetic — Subtracting Two Datetime Columns

**What:** When two columns are both `datetime64`, you can subtract one from the other to get the time difference.

**Returns:** A `timedelta64` Series — each value represents a duration.

**Getting the number of days:** Use `.dt.days` on the resulting timedelta Series.

**Syntax:**
```python
duration = df['end_date'] - df['start_date']
duration_in_days = duration.dt.days
```

In [ ]:
# Calculate delivery time in days
orders['delivery_days'] = (orders['delivery_date'] - orders['order_date']).dt.days

print("Delivery time per order:")
print(orders[['order_id', 'order_date', 'delivery_date', 'delivery_days']])

print("\nAverage delivery time (days):", orders['delivery_days'].mean().round(1))
print("Fastest delivery (days):     ", orders['delivery_days'].min())
print("Slowest delivery (days):     ", orders['delivery_days'].max())

---
### 3.6 Filtering by Date — Boolean Indexing with Dates

**What:** Filter rows using comparison operators (`>`, `<`, `==`) on datetime columns.

**Syntax:**
```python
# Filter by a specific date
df[df['date_col'] == '2024-01-15']

# Filter by month
df[df['date_col'].dt.month == 2]

# Filter a date range
df[(df['date_col'] >= '2024-01-01') & (df['date_col'] <= '2024-01-31')]
```

**Best practice:** Always convert the column to datetime first — comparing strings like `'2024-01-15'` without conversion can give wrong results.

In [ ]:
# Filter: orders placed in January 2024
jan_orders = orders[orders['order_date'].dt.month == 1]
print("January orders:")
print(jan_orders[['order_id', 'order_date', 'amount']])

# Filter: orders placed after February 2024
after_feb = orders[orders['order_date'] > '2024-02-28']
print("\nOrders after February 2024:")
print(after_feb[['order_id', 'order_date', 'amount']])

# Filter: orders in a date range (Jan)
jan_range = orders[
    (orders['order_date'] >= '2024-01-01') &
    (orders['order_date'] <= '2024-01-31')
]
print("\nOrders in January (range filter):")
print(jan_range[['order_id', 'order_date', 'amount']])

---
### ⚠️ Common Mistake: Using `.dt` on an Object (String) Column

**The mistake:** Trying to use `.dt.year` or `.dt.month` on a column that is still `object` dtype (string).

```python
# If 'order_date' is still a string (object dtype):
df['order_date'].dt.year   # AttributeError: Can only use .dt accessor with datetimelike values
```

**Fix:** Always convert to datetime first with `pd.to_datetime()`, then use `.dt`.

**How to check:** Run `df.dtypes` — datetime columns show `datetime64[ns]`, not `object`.

In [ ]:
# Demonstration of the wrong vs right approach
string_dates = pd.Series(['2024-01-15', '2024-02-10', '2024-03-05'])
print("dtype when stored as string:", string_dates.dtype)

# WRONG — uncomment to see the error
# string_dates.dt.year   # AttributeError!

# CORRECT — convert first, then use .dt
datetime_dates = pd.to_datetime(string_dates)
print("dtype after pd.to_datetime():", datetime_dates.dtype)
print("Years:", datetime_dates.dt.year.tolist())

---
## 4. Practical Examples

Now let's put it all together with realistic data-cleaning scenarios.

### 4.1 Cleaning a Messy Product Name Column

In [ ]:
products = pd.DataFrame({
    'product_name': ['  wireless MOUSE  ', 'USB-C CABLE', 'mechanical keyboard  ',
                     '  HDMI Adapter', 'webcam hd  '],
    'price': [799, 399, 2499, 599, 1299],
})

print("Before cleaning:")
print(products['product_name'].tolist())

# Step 1: strip whitespace
# Step 2: title case
# Step 3: replace '-' with ' ' for cleaner display
products['product_name_clean'] = (
    products['product_name']
    .str.strip()
    .str.title()
    .str.replace('-', ' ', regex=False)
)

print("\nAfter cleaning:")
display(products[['product_name', 'product_name_clean']])

### 4.2 Extracting Domain from Email Addresses

In [ ]:
# Using the employees dataset
# We already did the split — let's count domain frequencies
domain_counts = employees['domain'].value_counts()

print("Email domain distribution:")
display(domain_counts)

# Which employees use the most popular domain?
most_common_domain = domain_counts.index[0]
print(f"\nMost common domain: {most_common_domain}")
print("Employees using it:")
display(employees[employees['domain'] == most_common_domain][['name_clean', 'email']])

### 4.3 Calculating Age from Birthdate

In [ ]:
people = pd.DataFrame({
    'name':      ['Arun', 'Priya', 'Suresh', 'Meena'],
    'birthdate': ['1990-05-12', '1985-11-23', '2000-01-07', '1978-08-30'],
})

# Convert birthdate to datetime
people['birthdate'] = pd.to_datetime(people['birthdate'])

# Today's date as a datetime
today = pd.Timestamp('today').normalize()   # normalize() removes time portion
print("Today's date:", today.date())

# Calculate age in years (approximate: total days / 365.25)
people['age_days'] = (today - people['birthdate']).dt.days
people['age_years'] = (people['age_days'] / 365.25).astype(int)

print("\nPeople with calculated ages:")
print(people[['name', 'birthdate', 'age_years']])

### 4.4 Finding Which Day of Week Has Most Orders

In [ ]:
# Count orders by day-of-week name
day_counts = orders['weekday_name'].value_counts()

print("Order count by day of week:")
print(day_counts)

most_popular_day = day_counts.index[0]
print(f"\nMost orders placed on: {most_popular_day}")

### 4.5 Filtering Records from a Specific Month

In [52]:
# Show orders from February (month == 2)
feb_orders = orders[orders['order_date'].dt.month == 2]

print("February orders:")
display(feb_orders[['order_id', 'order_date', 'amount']])

print(f"\nTotal sales in February: ₹{feb_orders['amount'].sum():,}")
print(f"Number of orders:        {len(feb_orders)}")

February orders:


,order_id,order_date,amount
2,2,2024-02-10,3200
5,5,2024-02-22,2100



Total sales in February: ₹5,300
Number of orders:        2


---
## 5. Practice Questions

Work through these questions on your own before checking the answer key below.

---












**Q1 [Easy]** — You are given the DataFrame below. Clean the `full_name` column by:
1. Stripping whitespace
2. Converting to Title Case

```python

```

In [60]:
q1_df = pd.DataFrame({
    'full_name': ['  ravi kumar ', 'SUNITA SHARMA', 'ankit mehta  ', '  POOJA SINGH'],},
    index=[1,2,3,4])

display(q1_df)
display(q1_df.values.tolist())

q1_df['full_name'] = q1_df['full_name'].str.strip().str.title()

display(q1_df)



,full_name
1,ravi kumar
2,SUNITA SHARMA
3,ankit mehta
4,POOJA SINGH


[['  ravi kumar '], ['SUNITA SHARMA'], ['ankit mehta  '], ['  POOJA SINGH']]

,full_name
1,Ravi Kumar
2,Sunita Sharma
3,Ankit Mehta
4,Pooja Singh


---
**Q2 [Easy]** — From the `email` column below, extract just the username (the part before `@`).

```python

```

In [72]:
q2_df = pd.DataFrame({
    'email': ['ravi@gmail.com', 'sunita@yahoo.in', 'ankit@company.org', 'pooja@outlook.com'],},
    index=[1,2,3,4])

display(q2_df)

q2_df['User_Name'] = q2_df['email'].str.split('@').str[0].str.title()
display(q2_df)

,email
1,ravi@gmail.com
2,sunita@yahoo.in
3,ankit@company.org
4,pooja@outlook.com


,email,User_Name
1,ravi@gmail.com,Ravi
2,sunita@yahoo.in,Sunita
3,ankit@company.org,Ankit
4,pooja@outlook.com,Pooja


---
**Q3 [Easy]** — From the same `q2_df`, filter only the rows where the email ends with `'@gmail.com'`.

---

In [74]:
only_gmail = q2_df[q2_df['email'].str.endswith('@gmail.com')]
display(only_gmail)

,email,User_Name
1,ravi@gmail.com,Ravi


**Q4 [Easy]** — Convert the `signup_date` column below to datetime, then create new columns for `year`, `month`, and `day`.

```python

```

In [87]:
q4_df = pd.DataFrame({
    'user': ['A', 'B', 'C', 'D'],
    'signup_date': ['2023-06-01', '2023-11-15', '2024-01-22', '2024-05-30'],},
    index=[1,2,3,4])

q4_df['signup_date'] = pd.to_datetime(q4_df['signup_date'])
display(q4_df)

q4_df['year']  = q4_df['signup_date'].dt.year
q4_df['month'] = q4_df['signup_date'].dt.month
q4_df['day']   = q4_df['signup_date'].dt.day

display(q4_df)





,user,signup_date
1,A,2023-06-01
2,B,2023-11-15
3,C,2024-01-22
4,D,2024-05-30


,user,signup_date,year,month,day
1,A,2023-06-01,2023,6,1
2,B,2023-11-15,2023,11,15
3,C,2024-01-22,2024,1,22
4,D,2024-05-30,2024,5,30


---
**Q5 [Medium]** — Using `q4_df` (after converting to datetime), find how many users signed up in each year. Use `.value_counts()` on the `year` column.

---

In [ ]:
q4_df['year'].value_counts().to_frame(name='Year Wise')


,Year Wise
year,
2023,2
2024,2


**Q6 [Medium]** — Calculate the number of delivery days for each order in the DataFrame below. Also find the order with the fastest delivery.

```python

```

In [106]:
q6_df = pd.DataFrame({
    'order_id':      ['O1', 'O2', 'O3', 'O4'],
    'order_date':    ['2024-04-01', '2024-04-05', '2024-04-10', '2024-04-15'],
    'delivery_date': ['2024-04-04', '2024-04-07', '2024-04-16', '2024-04-17'],
},index=[1,2,3,4])

q6_df['delivery_date'] = pd.to_datetime(q6_df['delivery_date'])
q6_df['order_date']    = pd.to_datetime(q6_df['order_date'])

display(q6_df)

q6_df['delivery_days'] = q6_df['delivery_date'].dt.day - q6_df['order_date'].dt.day

display(q6_df)

print('fastest_delivery:')
q6_df[q6_df['delivery_days'] == q6_df['delivery_days'].min()]




,order_id,order_date,delivery_date
1,O1,2024-04-01,2024-04-04
2,O2,2024-04-05,2024-04-07
3,O3,2024-04-10,2024-04-16
4,O4,2024-04-15,2024-04-17


,order_id,order_date,delivery_date,delivery_days
1,O1,2024-04-01,2024-04-04,3
2,O2,2024-04-05,2024-04-07,2
3,O3,2024-04-10,2024-04-16,6
4,O4,2024-04-15,2024-04-17,2


fastest_delivery:


,order_id,order_date,delivery_date,delivery_days
2,O2,2024-04-05,2024-04-07,2
4,O4,2024-04-15,2024-04-17,2


---
**Q7 [Medium]** — You are given a raw product dataset. Build a full string cleaning pipeline:
1. Strip whitespace from `product_name`
2. Convert to Title Case
3. Replace `'&'` with `'and'`
4. Check which products contain `'Kit'` in the cleaned name

```python

```

In [115]:
q7_df = pd.DataFrame({
    'product_name': ['  paint & brush kit ', 'CANVAS BOARD', ' oil PASTEL kit  ',
                     'Watercolor & Palette', '  sketch book'],
    'price': [450, 200, 350, 500, 150],
},index=[1,2,3,4,5])

display(q7_df)

q7_df['product_name'] = q7_df['product_name'].str.strip().str.title().str.replace('&','and')

display(q7_df)


print('Kit Included:')

q7_df[q7_df['product_name'].str.contains('Kit')]

,product_name,price
1,paint & brush kit,450
2,CANVAS BOARD,200
3,oil PASTEL kit,350
4,Watercolor & Palette,500
5,sketch book,150


,product_name,price
1,Paint and Brush Kit,450
2,Canvas Board,200
3,Oil Pastel Kit,350
4,Watercolor and Palette,500
5,Sketch Book,150


Kit Included:


,product_name,price
1,Paint and Brush Kit,450
3,Oil Pastel Kit,350


---
## 6. Answer Key

Reveal one at a time — try each question first!

In [ ]:
# ── Answer Q1 ──────────────────────────────────────────────
q1_df = pd.DataFrame({
    'full_name': ['  ravi kumar ', 'SUNITA SHARMA', 'ankit mehta  ', '  POOJA SINGH'],
})

q1_df['name_clean'] = q1_df['full_name'].str.strip().str.title()

print("Q1 — Cleaned names:")
print(q1_df)

In [ ]:
# ── Answer Q2 ──────────────────────────────────────────────
q2_df = pd.DataFrame({
    'email': ['ravi@gmail.com', 'sunita@yahoo.in', 'ankit@company.org', 'pooja@outlook.com'],
})

q2_df['username'] = q2_df['email'].str.split('@').str[0]

print("Q2 — Usernames extracted:")
print(q2_df)

In [ ]:
# ── Answer Q3 ──────────────────────────────────────────────
gmail_mask = q2_df['email'].str.endswith('@gmail.com', na=False)
gmail_users = q2_df[gmail_mask]

print("Q3 — Gmail users only:")
print(gmail_users)

In [ ]:
# ── Answer Q4 ──────────────────────────────────────────────
q4_df = pd.DataFrame({
    'user': ['A', 'B', 'C', 'D'],
    'signup_date': ['2023-06-01', '2023-11-15', '2024-01-22', '2024-05-30'],
})

q4_df['signup_date'] = pd.to_datetime(q4_df['signup_date'])
q4_df['year']  = q4_df['signup_date'].dt.year
q4_df['month'] = q4_df['signup_date'].dt.month
q4_df['day']   = q4_df['signup_date'].dt.day

print("Q4 — Date parts extracted:")
print(q4_df)

In [ ]:
# ── Answer Q5 ──────────────────────────────────────────────
year_counts = q4_df['year'].value_counts().sort_index()

print("Q5 — Signups per year:")
print(year_counts)

most_signups_year = year_counts.idxmax()
print(f"\nYear with most signups: {most_signups_year}")

In [ ]:
# ── Answer Q6 ──────────────────────────────────────────────
q6_df = pd.DataFrame({
    'order_id':      ['O1', 'O2', 'O3', 'O4'],
    'order_date':    ['2024-04-01', '2024-04-05', '2024-04-10', '2024-04-15'],
    'delivery_date': ['2024-04-04', '2024-04-07', '2024-04-16', '2024-04-17'],
})

q6_df['order_date']    = pd.to_datetime(q6_df['order_date'])
q6_df['delivery_date'] = pd.to_datetime(q6_df['delivery_date'])

q6_df['delivery_days'] = (q6_df['delivery_date'] - q6_df['order_date']).dt.days

print("Q6 — Delivery days:")
print(q6_df)

fastest_idx = q6_df['delivery_days'].idxmin()
print(f"\nFastest delivery: Order {q6_df.loc[fastest_idx, 'order_id']} "
      f"({q6_df.loc[fastest_idx, 'delivery_days']} days)")

In [ ]:
# ── Answer Q7 ──────────────────────────────────────────────
q7_df = pd.DataFrame({
    'product_name': ['  paint & brush kit ', 'CANVAS BOARD', ' oil PASTEL kit  ',
                     'Watercolor & Palette', '  sketch book'],
    'price': [450, 200, 350, 500, 150],
})

# Step 1: strip → title → replace '&' with 'and'
q7_df['product_clean'] = (
    q7_df['product_name']
    .str.strip()
    .str.title()
    .str.replace('&', 'and', regex=False)
)

# Step 2: flag products containing 'Kit'
q7_df['is_kit'] = q7_df['product_clean'].str.contains('Kit', case=True, na=False)

print("Q7 — Full cleaning pipeline result:")
print(q7_df)

print("\nProducts that are kits:")
print(q7_df[q7_df['is_kit']][['product_clean', 'price']])

---
## 7. Quick Revision Table

| Operation | Code | Returns |
|-----------|------|---------|
| Strip whitespace (both sides) | `s.str.strip()` | Series of strings |
| Strip left only | `s.str.lstrip()` | Series of strings |
| Strip right only | `s.str.rstrip()` | Series of strings |
| Lowercase | `s.str.lower()` | Series of strings |
| Uppercase | `s.str.upper()` | Series of strings |
| Title Case | `s.str.title()` | Series of strings |
| Replace substring | `s.str.replace(old, new, regex=False)` | Series of strings |
| Check substring | `s.str.contains(pat, na=False)` | Boolean Series |
| Check prefix | `s.str.startswith(pre, na=False)` | Boolean Series |
| Check suffix | `s.str.endswith(suf, na=False)` | Boolean Series |
| Split string | `s.str.split(sep)` | Series of lists |
| Access split part | `s.str.split(sep).str[n]` | Series of strings |
| String length | `s.str.len()` | Numeric Series |
| String → datetime | `pd.to_datetime(s)` | datetime64 Series |
| Extract year | `s.dt.year` | Numeric Series |
| Extract month | `s.dt.month` | Numeric Series |
| Extract day | `s.dt.day` | Numeric Series |
| Extract hour | `s.dt.hour` | Numeric Series |
| Extract minute | `s.dt.minute` | Numeric Series |
| Day of week (int) | `s.dt.weekday` | 0=Mon … 6=Sun |
| Day of week (name) | `s.dt.day_name()` | Series of strings |
| Date only | `s.dt.date` | Series of date objects |
| Duration in days | `(end - start).dt.days` | Numeric Series |

---
## 8. Three Interview Questions

**Q1.** What is the `.str` accessor in Pandas? Why can't you just call `.upper()` directly on a Series?

> **Answer:** The `.str` accessor is a special namespace on a Pandas Series that lets you apply Python string methods element-wise to every value in the column. You can't call `.upper()` directly on a Series because `Series` objects don't have a `.upper()` method — only Python `str` objects do. The `.str` accessor bridges this gap.

---

**Q2.** What happens when you call `.str.strip()` on a column that has some `NaN` values?

> **Answer:** `NaN` values are propagated — the `.str.strip()` method returns `NaN` for those rows and does not raise an error. This is useful because you can still identify missing values after applying string operations. For methods like `.str.contains()`, you should explicitly set `na=False` or `na=True` to avoid getting `NaN` in a boolean mask that you plan to use for filtering.

---

**Q3.** What is the difference between `.dt.date` and `.dt.day`?

> **Answer:** `.dt.day` returns the **day of the month** as an integer (e.g., `15` for the 15th of any month). `.dt.date` returns a Python `date` object that represents the **full calendar date** (year, month, and day together) with the time component removed — useful when you want to group or display just the date without any time information.

---
## 9. What's Next?

**Notebook 5 — Sorting & Ranking**

Now that your data is cleaned and your dates are proper datetimes, the next step is to **sort and rank** your data meaningfully:

- `.sort_values()` — sort a DataFrame by one or more columns
- `.sort_index()` — sort by the index
- `.rank()` — assign ranks to values (handling ties)
- `ascending=True/False` — controlling sort direction
- Sorting by multiple columns
- Finding top-N and bottom-N rows

---
*Pandas from First Principles — Notebook 4 of 5*